In [1]:
import sys
from helpers import *
if ".." not in sys.path:
    sys.path.append("..")
    
import cobra
import pandas as pd
from cobra.io import read_sbml_model
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

model = read_sbml_model("../model/Rpom_05.xml")
model

Name,Rpom_05
Memory address,32fb179d0
Number of metabolites,1782
Number of reactions,1786
Number of genes,974
Number of groups,0
Objective expression,1.0*Rpom_hwa_biomass - 1.0*Rpom_hwa_biomass_reverse_5ec2f
Compartments,"c, p, e"


In [6]:
model.reactions.get_by_id("LIPID-RXN").reaction

'0.186220192 CPD-12819[c] + 0.478848946 CPD-12819[p] + 0.564392674 CPD-17086[c] + 0.219487198 CPD-17086[p] + 0.000573 UNDECAPRENYL-DIPHOSPHATE[c] --> LIPID[c]'

In [28]:
model.metabolites.get_by_id("CPD-17086[c]")

Metabolite identifier,CPD-17086[c]
Name,[]
Memory address,0x339979a00
Formula,C37H70N1O8P1
Compartment,c
In 3 reaction(s),"LIPID-RXN, 3.6.3.1-RXN, PHOSPHASERDECARB-RXN"


In [14]:
ATPM_space = np.linspace(0,50,100)
growth_rates = []
growth_rates_ac = []

import re
from collections import defaultdict

def _parse_formula(formula):
    d = defaultdict(int)
    for elem, num in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        d[elem] += int(num) if num else 1
    return d

def _dict_to_formula(d):
    order = ["C","H","N","O","P","S"]
    elems = order + [e for e in sorted(d.keys()) if e not in order]
    out = []
    for e in elems:
        n = d.get(e, 0)
        if n <= 0: continue
        out.append(e if n == 1 else f"{e}{n}")
    return "".join(out)

with model:
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    model.reactions.get_by_id("EX_ac").bounds = (0, 0)
    
    # key knockouts
    model.reactions.get_by_id("METHYLENETHFDEHYDROG-NADP-RXN").knock_out()
    model.reactions.get_by_id("RXN0-300").knock_out()
    model.reactions.get_by_id("GLYCERATE-DEHYDROGENASE-RXN").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NAD/WATER//OXALACETIC_ACID/AMMONIUM/NADH/PROTON.60.").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NADP/WATER//OXALACETIC_ACID/AMMONIUM/NADPH/PROTON.62.").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NAD/WATER//OXALACETIC_ACID/AMMONIUM/NADH/PROTON.60.").knock_out()
    model.reactions.get_by_id("NADH-KINASE-RXN").knock_out()
    model.reactions.get_by_id("R501-RXN-CPD-821/NADPH/OXYGEN-MOLECULE/PROTON//CPD-904/FORMALDEHYDE/NADP/WATER.70.").knock_out()
    model.reactions.get_by_id("R501-RXN-CPD-821/NADH/OXYGEN-MOLECULE/PROTON//CPD-904/FORMALDEHYDE/NAD/WATER.68.").knock_out()
    model.reactions.get_by_id("RXN0-3962").knock_out()
    
    # =====================================================================
    # COMMON METABOLITES
    # =====================================================================
    WATER = model.metabolites.get_by_id("WATER[c]")
    Pi = model.metabolites.get_by_id("Pi[c]")
    PROTON = model.metabolites.get_by_id("PROTON[c]")
    G3P = model.metabolites.get_by_id("GLYCEROL-3P[c]")
    CTP = model.metabolites.get_by_id("CTP[c]")
    PPI = model.metabolites.get_by_id("PPI[c]")
    CMP = model.metabolites.get_by_id("CMP[c]")
    SER = model.metabolites.get_by_id("SER[c]")
    CO2 = model.metabolites.get_by_id("CARBON-DIOXIDE[c]")
    COA = model.metabolites.get_by_id("CO-A[c]")
    ATP = model.metabolites.get_by_id("ATP[c]")
    ADP = model.metabolites.get_by_id("ADP[c]")
    NADH = model.metabolites.get_by_id("NADH[c]")
    NAD = model.metabolites.get_by_id("NAD[c]")
    O2 = model.metabolites.get_by_id("OXYGEN-MOLECULE[c]")
    orn = model.metabolites.get_by_id("L-ORNITHINE[c]")
    gln = model.metabolites.get_by_id("GLN[c]")
    ACP = model.metabolites.get_by_id("ACP[c]")
    
    # Acyl-CoA donors
    stearoyl_coa = model.metabolites.get_by_id("STEAROYL-COA[c]")
    oh_stearoyl_coa = model.metabolites.get_by_id("CPD-10261[c]")  # 3R-OH-C18:0-CoA
    vacc_coa = model.metabolites.get_by_id("CPD-18346[c]")         # cis-vaccenoyl-CoA
    
    # =====================================================================
    # KNOCK OUT OLD C16 GLYCEROPHOSPHOLIPID PATHWAY
    # =====================================================================
    old_rxns_to_ko = [
        "RXN_2.3.1.15", "RXN_2.3.1.51", "1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN",
        "CDPDIGLYSYN-RXN", "CDPDIGLYSYN-RXN-2",
        "PHOSPHASERSYN-RXN", "PHOSPHASERSYN-RXN-2",
        "PHOSPHASERDECARB-RXN", "PHOSPHASERDECARB-RXN-2",
        "3.6.3.1-RXN", "3.6.3.1-RXN-2",
        "PHOSPHAGLYPSYN-RXN", "LIPID-RXN",
    ]
    print("=== Knocking out old C16 pathway ===")
    for rid in old_rxns_to_ko:
        try:
            model.reactions.get_by_id(rid).knock_out()
            print(f"  KO: {rid}")
        except KeyError:
            print(f"  Skip: {rid}")
    
    # =====================================================================
    # FAS II FIXES — connect unsaturated branch to produce C18:1
    #
    # The model's FAS II pathway has the full unsaturated chain from C10
    # to C18:1 but two reactions are missing:
    #   1. FabA isomerase 5.3.3.14-RXN: trans-2-decenoyl-ACP ⇌ cis-3-decenoyl-ACP
    #      (branch point for unsaturated FA synthesis, EC 4.2.1.60)
    #   2. FabF elongation: palmitoleoyl-ACP + malonyl-ACP → 
    #      3-oxo-cis-vaccenoyl-ACP + ACP + CO2 (EC 2.3.1.179)
    #
    # We also add a thioesterase to release cis-vaccenate from the ACP
    # cycle, and a transport to move it to periplasm where the existing
    # CoA ligase (RXN0-7238) activates it to cis-vaccenoyl-CoA.
    #
    # Net route:
    #   FAS II → trans-2-decenoyl-ACP → FabA → cis-3-decenoyl-ACP
    #   → FabB elongation → ... → palmitoleoyl-ACP
    #   → FabF elongation → ... → cis-vaccenoyl-ACP
    #   → thioesterase → cis-vaccenate[c] + ACP (recycled)
    #   → transport → cis-vaccenate[p]
    #   → CoA ligase (RXN0-7238, exists) → cis-vaccenoyl-CoA
    # =====================================================================
    
    # FIX 1: FabA isomerase (EC 4.2.1.60)
    # trans-2-decenoyl-ACP ⇌ cis-3-decenoyl-ACP
    t2d = model.metabolites.get_by_id("Trans-D2-decenoyl-ACPs[c]")
    c3d = model.metabolites.get_by_id("Cis-delta-3-decenoyl-ACPs[c]")
    
    # this exists in R Pom
    # bifunctional 3-hydroxydecanoyl-ACP dehydratase/trans-2-decenoyl-ACP isomerase
    # NCBI Reference Sequence: WP_011242221.1
    rxn_fabA = cobra.Reaction("5.3.3.14-RXN")
    rxn_fabA.name = "5.3.3.14-RXN"
    rxn_fabA.bounds = (-1000.0, 1000.0)
    rxn_fabA.add_metabolites({t2d: -1, c3d: 1})
    
    # FIX 2: FabF elongation of C16:1 → C18:1 (EC 2.3.1.179)
    # palmitoleoyl-ACP + malonyl-ACP + H+ → 3-oxo-cis-vaccenoyl-ACP + ACP + CO2
    
    # this exists in R Pom
    # beta-ketoacyl-ACP synthase II
    # NCBI Reference Sequence: WP_011047991.1
    palm_oleyl = model.metabolites.get_by_id("Palmitoleoyl-ACPs[c]")
    mal_acp = model.metabolites.get_by_id("MALONYL-ACP[c]")
    oxo_vacc = model.metabolites.get_by_id("3-oxo-cis-vaccenoyl-ACPs[c]")
    
    rxn_fabF = cobra.Reaction("3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP")
    rxn_fabF.name = "3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP"
    rxn_fabF.bounds = (0, 1000.0)
    rxn_fabF.add_metabolites({
        palm_oleyl: -1, mal_acp: -1, PROTON: -1,
        oxo_vacc: 1, ACP: 1, CO2: 1,
    })
    
    # FIX 3: cis-vaccenoyl-ACP thioesterase (EC 3.1.2.14)
    # cis-vaccenoyl-ACP + H2O → cis-vaccenate + ACP + H+
    
    # reaction likely exists
    # acyl-CoA thioesterase
    # NCBI Reference Sequence: WP_011047689.1
    vacc_acp_met = model.metabolites.get_by_id("Cis-vaccenoyl-ACPs[c]")
    
    vacc_c = cobra.Metabolite("CPD-9247[c]", formula="C18H33O2", charge=-1,
        name="cis-vaccenate", compartment="c")
    
    rxn_vacc_TE = cobra.Reaction("RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON")
    rxn_vacc_TE.name = "RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON"
    rxn_vacc_TE.bounds = (0, 1000.0)
    rxn_vacc_TE.add_metabolites({
        vacc_acp_met: -1, WATER: -1,
        ACP: 1, vacc_c: 1, PROTON: 1,
    })
    
    # FIX 4:
    # cis-vaccenate[c] + ATP + CoA → cis-vaccenoyl-CoA + AMP + PPi
    
    # reaction likely exists
    # acyl-CoA synthetase
    # NCBI Reference Sequence: WP_011046428.1
    AMP = model.metabolites.get_by_id("AMP[c]")
    
    rxn_vacc_ligase = cobra.Reaction("RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c]")
    rxn_vacc_ligase.name = "RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c]"
    rxn_vacc_ligase.bounds = (0, 1000.0)
    rxn_vacc_ligase.add_metabolites({
        vacc_c: -1, ATP: -1, COA: -1,
        vacc_coa: 1, AMP: 1, PPI: 1,
    })
    
    print("\n=== FAS II FIXES — MASS BALANCE ===")
    fas2_rxns = [rxn_fabA, rxn_fabF, rxn_vacc_TE, rxn_vacc_ligase]
    for rxn in fas2_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:40s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
    model.add_reactions(fas2_rxns)
    
    
    
    
    # =====================================================================
    # C18:1 GLYCEROPHOSPHOLIPID and PHOSPHATIDYLETHANOLAMINE PATHWAY (CoA-based)
    # https://metacyc.org/pathway?orgid=META&id=PWY4FS-8
    # Uses cis-vaccenoyl-CoA (CPD-18346) for both PlsB and PlsC.
    # All formulas derived computationally from mass balance:
    #   lyso-PA  = C21H39O7P      charge=-2
    #   PA       = C39H71O8P      charge=-2
    #   CDP-DAG  = C48H83N3O15P2  charge=-2
    #   PS       = C42H77NO10P    charge=-1
    #   PE       = C41H78NO8P     charge=0
    #   PGP      = C42H77O13P2    charge=-3
    #   PG       = C42H77O10P     charge=-2
    # =====================================================================
    
    # inferred metabolites
    lyso_PA = cobra.Metabolite("LYSO-PA-C18[c]", formula="C21H39O7P", charge=-2,
        name="1-(11Z)-vaccenoyl-sn-glycerol-3-phosphate", compartment="c")
    PA = cobra.Metabolite("PA-C18[c]", formula="C39H71O8P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-sn-glycerol-3-phosphate", compartment="c")
    CDP_DAG = cobra.Metabolite("CDP-DAG-C18[c]", formula="C48H83N3O15P2", charge=-2,
        name="CDP-1,2-di-(11Z)-vaccenoyl-glycerol", compartment="c")
    PS = cobra.Metabolite("PS-C18[c]", formula="C42H77NO10P", charge=-1,
        name="1,2-di-(11Z)-vaccenoyl-phosphatidylserine", compartment="c")
    PE_c = cobra.Metabolite("PE-C18[c]", formula="C41H78NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-PE", compartment="c")
    PE_p = cobra.Metabolite("PE-C18[p]", formula="C41H78NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-PE", compartment="p")
    PGP_ = cobra.Metabolite("PGP-C18[c]", formula="C42H77O13P2", charge=-3,
        name="1,2-di-(11Z)-vaccenoyl-PGP", compartment="c")
    PG = cobra.Metabolite("PG-C18[c]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="c")
    PG_e = cobra.Metabolite("PG-C18[e]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="e")
    
    # RXN-1381 
    # an acyl-CoA + sn-glycerol 3-phosphate → a 2-lysophosphatidate + coenzyme A
    rxn_plsB = cobra.Reaction("RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A")
    rxn_plsB.name = "RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A"
    rxn_plsB.bounds = (0, 1000.0)
    rxn_plsB.add_metabolites({G3P: -1, vacc_coa: -1, lyso_PA: 1, COA: 1})
    
    rxn_plsC = cobra.Reaction("RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A")
    rxn_plsC.name = "RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A"
    rxn_plsC.bounds = (0, 1000.0)
    rxn_plsC.add_metabolites({lyso_PA: -1, vacc_coa: -1, PA: 1, COA: 1})
    
    rxn_cdsA = cobra.Reaction("CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI")
    rxn_cdsA.name = "CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI"
    rxn_cdsA.bounds = (0, 1000.0)
    rxn_cdsA.add_metabolites({PA: -1, CTP: -1, PROTON: -1, CDP_DAG: 1, PPI: 1})
    
    # ============================
    # Phosphatidylethanolamine synthesis leveraging same pathway from before 
    # using phosphatidylserine precursor but now using 18 carbons 
    rxn_pssA = cobra.Reaction("PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON")
    rxn_pssA.name = "PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON"
    rxn_pssA.bounds = (0, 1000.0)
    rxn_pssA.add_metabolites({CDP_DAG: -1, SER: -1, PS: 1, CMP: 1, PROTON: 1})
    
    rxn_psd = cobra.Reaction("PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE")
    rxn_psd.name = "PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE"
    rxn_psd.bounds = (0, 1000.0)
    rxn_psd.add_metabolites({PS: -1, PROTON: -1, PE_c: 1, CO2: 1})
    
    rxn_flip = cobra.Reaction("3.6.3.1-RXN-PE-C18[c]/ATP/WATER//PE-C18[p]/ADP/PROTON/Pi")
    rxn_flip.name = "3.6.3.1-RXN-PE-C18[c]/ATP/WATER//PE-C18[p]/ADP/PROTON/Pi"
    rxn_flip.bounds = (0, 1000.0)
    rxn_flip.add_metabolites({ATP: -1, PE_c: -1, WATER: -1,
                               ADP: 1, PE_p: 1, PROTON: 1, Pi: 1})
    
    # ============================
    # phosphatidyl glycerol synthesis continued
    rxn_pgsA = cobra.Reaction("PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON")
    rxn_pgsA.name = "PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON"
    rxn_pgsA.bounds = (0, 1000.0)
    rxn_pgsA.add_metabolites({CDP_DAG: -1, G3P: -1, PGP_: 1, CMP: 1, PROTON: 1})
    
    rxn_pgp = cobra.Reaction("PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON")
    rxn_pgp.name = "PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON"
    rxn_pgp.bounds = (0, 1000.0)
    rxn_pgp.add_metabolites({PGP_: -1, WATER: -1, PG: 1, Pi: 1, PROTON: 1})
    
    glycerophospholipid_rxns = [rxn_plsB, rxn_plsC, rxn_cdsA, rxn_pssA,
                                 rxn_psd, rxn_flip, rxn_pgsA, rxn_pgp]
    model.add_reactions(glycerophospholipid_rxns)
    
    print("\n=== C18:1 GLYCEROPHOSPHOLIPID — MASS BALANCE ===")
    for rxn in glycerophospholipid_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:35s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    # =====================================================================
    # ORNITHINE LIPID (OL) — mixed chains: 3-OH-C18:0 (amide) + C18:1 (ester)
    # MetaCyc PWY-6818. R. pomeroyi: SPO1980=OlsB, SPO1979=OlsA
    # References: Weissenmayer 2002, Gao 2004, Smith 2019, Stirrup 2022
    #
    # OlsB uses 3-OH-stearoyl-CoA (CPD-10261, producible, C18:0).
    # OlsA uses vaccenoyl-CoA (CPD-18346, C18:1).
    # Mixed chains (saturated amide + unsaturated ester) common in bacteria.
    #
    # Formulas from mass balance:
    #   lyso-OL = orn + CPD-10261 - CoA - H+ = C23H46N2O4, charge 0
    #   OL = lyso-OL + CPD-18346 - CoA = C41H78N2O5, charge 0
    
    # SEE WRAP_Theses_Silvano_2021.pdf
    # https://www.nature.com/articles/s41396-018-0249-z/figures/2
    # =====================================================================
    
    # =====================================================================
    # FREE 3-HYDROXY-STEARIC ACID (new metabolite)
    # Released from 3-OH-stearoyl-CoA by thioesterase
    # Formula: CPD-10261 (C39H66N7O18P3S) + H2O - CoA (C21H32N7O16P3S) - H+
    #        = C18H35O3, charge -1
    # =====================================================================
    
    OH_STEARATE = cobra.Metabolite("3R-OH-STEARATE[c]",
        formula="C18H35O3", charge=-1,
        name="(3R)-3-hydroxystearate (free acid)", compartment="c")
    lyso_OL = cobra.Metabolite("LYSO-OL-C23[c]",
        formula="C23H46N2O4", charge=0,
        name="(2R)-5-amino-2-(3-hydroxyoctadecanoylamino)pentanoic acid", compartment="c")
    OL = cobra.Metabolite("OL-LIPID-C41[c]",
        formula="C41H77N2O5", charge=-1,
        name="ornithine lipid (3-OH-C18:0 amide, C18:1 ester)", compartment="c")
    
    # Thioesterase: 3-OH-stearoyl-CoA + H2O → 3-OH-stearate + CoA + H+
    # Analogous to RXN-9624 (stearoyl-CoA thioesterase)
    # likely real reaction
    # Gene: WP_011047689.1 (acyl-CoA thioesterase, broad specificity)
    rxn_oh_stearate_TE = cobra.Reaction("THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON")
    rxn_oh_stearate_TE.name = "THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON"
    rxn_oh_stearate_TE.bounds = (0, 1000.0)
    rxn_oh_stearate_TE.add_metabolites({
        oh_stearoyl_coa: -1, WATER: -1,
        OH_STEARATE: 1, COA: 1, PROTON: 1,
    })
    
    model.add_reactions([rxn_oh_stearate_TE])
    bal = rxn_oh_stearate_TE.check_mass_balance()
    print(f"  Thioesterase mass balance: {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    rxn_olsB = cobra.Reaction("RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER")
    rxn_olsB.name = "RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER"
    rxn_olsB.bounds = (0, 1000.0)
    rxn_olsB.add_metabolites({
        orn: -1, OH_STEARATE: -1,
        lyso_OL: 1, WATER: 1,
    })
    
    rxn_olsA_OL = cobra.Reaction("RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER")
    rxn_olsA_OL.name = "RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER"
    rxn_olsA_OL.bounds = (0, 1000.0)
    rxn_olsA_OL.add_metabolites({
        lyso_OL: -1, vacc_c: -1,
        OL: 1, WATER: 1,
    })
    
    # =====================================================================
    # GLUTAMINE LIPID (QL) — free fatty acid route
    # GlsB: glutamine + 3-OH-stearate → lyso-QL + H2O
    # OlsA: lyso-QL + cis-vaccenate → QL + H2O
    # see https://www.nature.com/articles/s41396-018-0249-z/figures/2
    # =====================================================================
    
    lyso_QL = cobra.Metabolite("LYSO-QL-C23[c]",
        formula="C23H43N2O5", charge=-1,
        name="lyso-QL (N-(3R-hydroxyoctadecanoyl)-L-glutamine)", compartment="c")
    
    QL = cobra.Metabolite("QL-LIPID-C41[c]",
        formula="C41H74N2O6", charge=-2,
        name="glutamine lipid (3-OH-C18:0 amide, C18:1 ester)", compartment="c")
    
    rxn_glsB = cobra.Reaction("RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER")
    rxn_glsB.name = "RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER"
    rxn_glsB.bounds = (0, 1000.0)
    rxn_glsB.add_metabolites({
        gln: -1, OH_STEARATE: -1,
        lyso_QL: 1, WATER: 1,
    })
    
    rxn_olsA_QL = cobra.Reaction("RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER")
    rxn_olsA_QL.name = "RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER"
    rxn_olsA_QL.bounds = (0, 1000.0)
    rxn_olsA_QL.add_metabolites({
        lyso_QL: -1, vacc_c: -1,
        QL: 1, WATER: 1,
    })
    
    aminolipid_rxns = [rxn_olsB, rxn_olsA_OL, rxn_glsB, rxn_olsA_QL]
    model.add_reactions(aminolipid_rxns)
    
    print("\n=== AMINOLIPID (free acid route) — MASS BALANCE ===")
    for rxn in [rxn_oh_stearate_TE] + aminolipid_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:35s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    # =====================================================================
    # SAL — SULFUR-CONTAINING AMINOLIPID
    # Smith et al. 2021 ISME J 15:2440-53. SalA=SPO0716.
    # Silvano 2021 thesis Fig 4.7 (pathway diagram).
    #
    # Headgroup = aminopropanesulfonate (APS; homotaurine or 2-APS)
    # Lumped APS synthesis from homocysteine.
    # SalB (unknown N-acyltransferase): APS + 3-OH-stearoyl-CoA → lyso-SAL
    # SalA (SPO0716): lyso-SAL + vaccenoyl-CoA → SAL
    # Mixed chains like OL/QL.
    # =====================================================================
    
    CYSTEATE = model.metabolites.get_by_id("L-CYSTEATE[c]")
    
    LYSO_SAL = cobra.Metabolite("LYSO-SAL[c]",
        formula="C21H39NO7S", charge=-2,
        name="lyso-SAL (N-(3R-hydroxyoctadecanoyl)-L-cysteate)", compartment="c")
    
    SAL = cobra.Metabolite("SAL-LIPID[c]",
        formula="C39H70NO8S", charge=-3,
        name="SAL (3-OH-C18:0 amide, C18:1 ester, cysteate headgroup)", compartment="c")
    
    # SalB: cysteate + 3-OH-stearate → lyso-SAL + H2O
    rxn_salB = cobra.Reaction("RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER")
    rxn_salB.name = "RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER (unknown enzyme)"
    rxn_salB.bounds = (0, 1000.0)
    rxn_salB.add_metabolites({
        CYSTEATE: -1, OH_STEARATE: -1,
        LYSO_SAL: 1, WATER: 1,
    })
    
    # SalA: lyso-SAL + cis-vaccenate → SAL + H2O
    rxn_salA = cobra.Reaction("RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER")
    rxn_salA.name = "RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER (SPO0716)"
    rxn_salA.bounds = (0, 1000.0)
    rxn_salA.add_metabolites({
        LYSO_SAL: -1, vacc_c: -1,
        SAL: 1, WATER: 1,
    })
    
    sal_rxns = [rxn_salB, rxn_salA] 
    model.add_reactions(sal_rxns)
    
    print("\n=== SAL PATHWAY — MASS BALANCE ===")
    for rxn in sal_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
    # =====================================================================
    # N-METHYLATION PATHWAY
    # For DMPE and MMPE generation
    # =====================================================================
    
    # PE N-methylation pathway (PmtA, EC 2.1.1.17)
    # R. pomeroyi accumulates MMPE and DMPE (Stirrup 2022)
    # Uses SAM as methyl donor, produces SAH
    
    SAM = model.metabolites.get_by_id("S-ADENOSYLMETHIONINE[c]")
    SAH = model.metabolites.get_by_id("ADENOSYL-HOMO-CYS[c]")
    
    # MMPE = PE + CH3 (from SAM) - H (replaced on nitrogen)
    # PE:   C41H78NO8P, charge=0
    # +CH3, -H = +CH2 = +C +2H
    # MMPE: C42H80NO8P, charge=0
    MMPE_c = cobra.Metabolite("MMPE-C18[c]",
        formula="C42H80NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-MMPE", compartment="c")
    
    # DMPE = MMPE + CH2
    # DMPE: C43H82NO8P, charge=0
    DMPE_c = cobra.Metabolite("DMPE-C18[c]",
        formula="C43H82NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-DMPE", compartment="c")
    
    # Step 1: PE + SAM → MMPE + SAH + H+
    rxn_pmt1 = cobra.Reaction("2.1.1.17-RXN-PE-C18/SAM//MMPE-C18/SAH/PROTON")
    rxn_pmt1.name = "2.1.1.17-RXN-PE-C18/SAM//MMPE-C18/SAH/PROTON"
    # gene SPOA0294
    rxn_pmt1.bounds = (0, 1000.0)
    rxn_pmt1.add_metabolites({
        PE_c: -1, SAM: -1,
        MMPE_c: 1, SAH: 1, PROTON: 1,
    })
    # Step 2: MMPE + SAM → DMPE + SAH + H+
    rxn_pmt2 = cobra.Reaction("2.1.1.71-RXN-MMPE-C18/SAM//DMPE-C18/SAH/PROTON")
    rxn_pmt2.name = "2.1.1.71-RXN-MMPE-C18/SAM//DMPE-C18/SAH/PROTON"
    rxn_pmt2.bounds = (0, 1000.0)
    rxn_pmt2.add_metabolites({
        MMPE_c: -1, SAM: -1,
        DMPE_c: 1, SAH: 1, PROTON: 1,
    })
    
    n_methylation_rxns = [rxn_pmt1, rxn_pmt2]
    model.add_reactions(n_methylation_rxns)
    
    print("\n=== N-METHYLATION PATHWAY — MASS BALANCE ===")
    for rxn in n_methylation_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
       
    # =====================================================================
    # LIPID TRANSPORT — inner membrane to outer membrane
    # MsbA (EC 3.6.3.1) flips lipids across the inner membrane.
    # Consistent with existing model reactions 3.6.3.1-RXN and 3.6.3.1-RXN-2
    # which use: lipid[c] + ATP + H2O → lipid[p] + ADP + Pi + H+
    #
    # We already have PE[c]→[p] flippase (rxn_flip).
    # Add PG, OL, QL, SAL flippases using the same stoichiometry.
    # 
    # Gene: MsbA homolog likely present in R. pomeroyi (essential in 
    # gram-negatives). MsbA has broad substrate specificity for lipids.
    # =====================================================================
    
    MMPE_p = cobra.Metabolite("MMPE-C18[p]", formula=MMPE_c.formula, charge=MMPE_c.charge,
        name=MMPE_c.name, compartment="p")
    DMPE_p = cobra.Metabolite("DMPE-C18[p]", formula=DMPE_c.formula, charge=DMPE_c.charge,
        name=DMPE_c.name, compartment="p")
    
    rxn_flip_m = cobra.Reaction("3.6.3.1-RXN-MMPE-C18[c]/ATP/WATER//MMPE-C18[p]/ADP/PROTON/Pi")
    rxn_flip_m.name = "3.6.3.1-RXN-MMPE-C18[c]/ATP/WATER//MMPE-C18[p]/ADP/PROTON/Pi"
    rxn_flip_m.bounds = (0, 1000.0)
    rxn_flip_m.add_metabolites({ATP: -1, MMPE_c: -1, WATER: -1,
                               ADP: 1, MMPE_p: 1, PROTON: 1, Pi: 1})
    
    rxn_flip_d = cobra.Reaction("3.6.3.1-RXN-DMPE-C18[c]/ATP/WATER//DMPE-C18[p]/ADP/PROTON/Pi")
    rxn_flip_d.name = "3.6.3.1-RXN-DMPE-C18[c]/ATP/WATER//DMPE-C18[p]/ADP/PROTON/Pi"
    rxn_flip_d.bounds = (0, 1000.0)
    rxn_flip_d.add_metabolites({ATP: -1, DMPE_c: -1, WATER: -1,
                               ADP: 1, DMPE_p: 1, PROTON: 1, Pi: 1})
    
    # Metacyc derived
    
    # PG flippase [c] → [p]
    PG_p = cobra.Metabolite("PG-C18[p]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="p")
        
    rxn_pg_flip = cobra.Reaction("TRANS-RXN-1533-PG-C18[c]/PG-C18[e]")
    # no known enzyme
    rxn_pg_flip.name = "TRANS-RXN-1533-PG-C18[c]/PG-C18[e]"
    rxn_pg_flip.bounds = (0, 1000.0)
    rxn_pg_flip.add_metabolites({
        PG: -1,
        PG_p: 1
    })
    
    # OL flippase [c] → [p]
    OL_p = cobra.Metabolite("OL-LIPID-C41[p]", formula=OL.formula, charge=OL.charge,
        name=OL.name, compartment="p")
    # yddG gene in E. Coli.
    # but not known in R Pom. Yddg BLAST returns no results.
    rxn_ol_flip = cobra.Reaction("TRANS-RXN0-265-OL-C41[c]//OL-C41[p]")
    rxn_ol_flip.name = "TRANS-RXN0-265-OL-C41[c]//OL-C41[p]"
    rxn_ol_flip.bounds = (0, 1000.0)
    rxn_ol_flip.add_metabolites({
        OL: -1,
        OL_p: 1
    })
    
    # QL flippase [c] → [p]
    QL_p = cobra.Metabolite("QL-LIPID-C41[p]", formula=QL.formula, charge=QL.charge,
        name=QL.name, compartment="p")
    
    rxn_ql_flip = cobra.Reaction("TRANS-RXN0-265-QL-C41[c]//QL-C41[p]")
    rxn_ql_flip.name = "TRANS-RXN0-265-QL-C41[c]//QL-C41[p]"
    rxn_ql_flip.bounds = (0, 1000.0)
    rxn_ql_flip.add_metabolites({
        QL: -1, 
        QL_p: 1
    })
    
    # SAL flippase [c] → [p]
    SAL_p = cobra.Metabolite("SAL-LIPID[p]", formula=SAL.formula, charge=SAL.charge,
        name=SAL.name, compartment="p")
    
    rxn_sal_flip = cobra.Reaction("TRANS-RXN0-265-SAL[c]//SAL[p]")
    rxn_sal_flip.name = "TRANS-RXN0-265-SAL[c]//SAL[p]"
    rxn_sal_flip.bounds = (0, 1000.0)
    rxn_sal_flip.add_metabolites({
        SAL: -1,
        SAL_p: 1
    })
    
    flippase_rxns = [rxn_pg_flip, rxn_ol_flip, rxn_ql_flip, rxn_sal_flip, rxn_flip_m, rxn_flip_d]
    model.add_reactions(flippase_rxns)
    
    print("\n=== FLIP REACTIONS — MASS BALANCE ===")
    for rxn in flippase_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
        
    # =====================================================================
    # LYSO-PE SYNTHESIS PATHWAY
    # =====================================================================
    WATER_p = model.metabolites.get_by_id("WATER[p]")
    PROTON_p = model.metabolites.get_by_id("PROTON[p]")
    
    vacc_p = cobra.Metabolite("CPD-9247[p]", formula="C18H33O2", charge=-1,
        name="cis-vaccenate", compartment="p")
    
    lyso_PE_p = cobra.Metabolite("LYSO-PE-C18[p]",
        formula="C23H46NO7P", charge=0,
        name="1-(11Z)-vaccenoyl-sn-glycero-3-phosphoethanolamine", compartment="p")
    
    lyso_PE_c = cobra.Metabolite("LYSO-PE-C18[c]",
        formula="C23H46NO7P", charge=0,
        name="1-(11Z)-vaccenoyl-sn-glycero-3-phosphoethanolamine", compartment="c")
    
    # PldA: outer membrane phospholipase A (EC 3.1.1.32)
    # PE[p] + H2O[p] → lyso-PE[p] + vaccenate[p] + H+[p]
    # not in biocyc
    # e coli pldA
    rxn_pldA = cobra.Reaction("RXN0-6952-PE-C18[p]/WATER[p]//LYSO-PE-C18[p]/CPD-9247[p]/PROTON[p]")
    rxn_pldA.name = "RXN0-6952-PE-C18[p]/WATER[p]//LYSO-PE-C18[p]/CPD-9247[p]/PROTON[p]"
    rxn_pldA.bounds = (0, 1000.0)
    rxn_pldA.add_metabolites({
        PE_p: -1, WATER_p: -1,
        lyso_PE_p: 1, vacc_p: 1, PROTON_p: 1,
    })
    
    # LplT: lysophospholipid transporter (energy-independent)
    # lyso-PE[p] → lyso-PE[c]
    # not in biocyc
    # e coli lplT
    rxn_lplT = cobra.Reaction("TRANS-RXN-294-LYSO-PE-C18[p]//LYSO-PE-C18[c]")
    rxn_lplT.name = "TRANS-RXN-294-LYSO-PE-C18[p]//LYSO-PE-C18[c]"
    rxn_lplT.bounds = (0, 1000.0)
    rxn_lplT.add_metabolites({
        lyso_PE_p: -1,
        lyso_PE_c: 1,
    })
    
    lyso_pe_rxns = [rxn_pldA, rxn_lplT]
    model.add_reactions(lyso_pe_rxns)
    
    print("\n=== LYSO-PE PATHWAY (PldA/LplT/Aas) — MASS BALANCE ===")
    for rxn in lyso_pe_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:55s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    
    # =====================================================================
    # LIPID BIOMASS PSEUDOREACTION
    # =====================================================================
    
    lipid_rxn = cobra.Reaction("LIPID-RXN-RPOM")
    lipid_rxn.name = "Lipid pseudoreaction R. pomeroyi (Stirrup 2022, IM/OM resolved)"
    lipid_rxn.bounds = (0, 1000.0)
    """
    lipid_rxn.add_metabolites({
        PG:         -0.820508499,
        PG_p:       -0.549169986,
        PE_c:       -10.88065619,
        PE_p:       -4.207200571,
        MMPE_c:     -4.004081477,
        MMPE_p:     -1.286139035,
        DMPE_c:     -50.05101847,
        DMPE_p:     -15.5140521,
        lyso_PE_c:  -100.1020369,
        lyso_PE_p:  -82.74161122,
        OL:         -7.470301264,
        OL_p:       -9.547108987,
        QL:         -3.653359012,
        QL_p:       -1.021501373,
        SAL:        -0.0,
        SAL_p:      -31.02810421,
        lipid_met:   1.0
    })
    """
    lipid_rxn.add_metabolites({
        PG:         -1.8461,
        PG_p:       -4.9425,
        PE_c:       -24.4815,
        PE_p:       -37.8648,
        MMPE_c:     -9.0092,
        MMPE_p:     -11.5753,
        DMPE_c:     -112.6148,
        DMPE_p:     -139.6265,
        lyso_PE_c:  -225.2296,
        lyso_PE_p:  -744.6745,
        OL:         -16.8082,
        OL_p:       -85.9240,
        QL:         -8.2201,
        QL_p:       -9.1935,
        # SAL_c: 0
        SAL_p:      -279.2529,
        lipid_met:   1.0
    })
    model.add_reactions([lipid_rxn])
    
    print(f"\n=== LIPID-RXN-RPOM (IM/OM resolved) ===")
    print(f"  {'Species':12s} {'IM ([c])':>12s} {'OM ([p])':>12s} {'Total':>12s}")
    print(f"  {'-'*48}")
    for label, met_c, met_p, coeff_c, coeff_p in [
        ("PG",      PG,        PG_p,       0.820508499, 0.549169986),
        ("PE",      PE_c,      PE_p,       10.88065619,  4.207200571),
        ("MMPE",    MMPE_c,    MMPE_p,     4.004081477,  1.286139035),
        ("DMPE",    DMPE_c,    DMPE_p,     50.05101847,  15.5140521),
        ("lyso-PE", lyso_PE_c, lyso_PE_p,  100.1020369,  82.74161122),
        ("OL",      OL,        OL_p,       7.470301264,  9.547108987),
        ("QL",      QL,        QL_p,       3.653359012,  1.021501373),
        ("SAL",     SAL,       SAL_p,      0.0,          31.02810421),
    ]:
        print(f"  {label:12s} {coeff_c:>12.4f} {coeff_p:>12.4f} {coeff_c+coeff_p:>12.4f}")    
    
    # =====================================================================
    # Fix NADH dehydrogenase (e → p)
    # =====================================================================
    model.reactions.get_by_id("NADH-DEHYDROG-A-RXN").knock_out()
    rxn_ndh = cobra.Reaction("NADH-DEHYDROG-A-RXN_2")
    rxn_ndh.add_metabolites({
        NADH: -1, model.metabolites.get_by_id("PROTON[c]"): -5,
        model.metabolites.get_by_id("UBIQUINONE-10[c]"): -1,
        model.metabolites.get_by_id("CPD-9958[c]"): 1, NAD: 1,
        model.metabolites.get_by_id("PROTON[p]"): 4,
    })
    rxn_ndh.bounds = (-1000, 1000)
    model.add_reactions([rxn_ndh])
    
    # =====================================================================
    # FULL VALIDATION
    # =====================================================================
    print("\n=== FULL MASS BALANCE CHECK ===")
    all_new = fas2_rxns + glycerophospholipid_rxns + aminolipid_rxns + sal_rxns
    all_ok = True
    for rxn in all_new:
        bal = rxn.check_mass_balance()
        if bal: all_ok = False
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    print(f"\n  {'ALL BALANCED!' if all_ok else 'WARNING: imbalances'}")
    
    # =====================================================================
    # TEST GROWTH
    # =====================================================================
    model.objective = "Rpom_hwa_biomass"
    
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    model.reactions.get_by_id("EX_ac").bounds = (0, 0)
    sol_glc = model.optimize()
    print(f"\nGlucose growth: {sol_glc.objective_value:.4f}")
    
    model.reactions.get_by_id("EX_glc").bounds = (0, 0)
    model.reactions.get_by_id("EX_ac").bounds = (-15.01, 0)
    sol_ac = model.optimize()
    print(f"Acetate growth:  {sol_ac.objective_value:.4f}")
    
    if sol_glc.objective_value > 0 and sol_ac.objective_value > 0:
        print(f"Ratio (glc/ac):  {sol_glc.objective_value / sol_ac.objective_value:.2f}")
        
        model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
        model.reactions.get_by_id("EX_ac").bounds = (0, 0)
        sol = model.optimize()
        print(f"\n=== Key lipid fluxes (glucose) ===")
        for rid in [
            "5.3.3.14-RXN",
            "3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP",
            "RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON",
            "RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c]",
            "RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A",
            "RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A",
            "CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI",
            "PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON",
            "PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE",
            "PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON",
            "PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON",
            "THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON",
            "RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER",
            "RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER",
            "RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER",
            "RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER",
            "RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER",
            "RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER",
        ]:
            print(f"  {rid[:50]:50s} {sol.fluxes[rid]:.6f}")
        
        # ATPM scans
        for level in tqdm(ATPM_space):
            model.reactions.get_by_id("ATPM").bounds = (level, level)
            model.reactions.get_by_id("EX_ac").bounds = (0, 0)
            model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
            sol = model.optimize()
            growth_rates.append(sol.objective_value)
        
        for level in tqdm(ATPM_space):
            model.reactions.get_by_id("ATPM").bounds = (level, level)
            model.reactions.get_by_id("EX_glc").bounds = (0, 0)
            model.reactions.get_by_id("EX_ac").bounds = (-15.01, 0)
            sol = model.optimize()
            growth_rates_ac.append(sol.objective_value)
    else:
        print("\n=== ZERO GROWTH DEBUG ===")
        model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
        model.reactions.get_by_id("EX_ac").bounds = (0, 0)
        
        lipid_rxn.bounds = (0, 0)
        sol = model.optimize()
        print(f"  Without lipid req: {sol.objective_value:.4f}")
        lipid_rxn.bounds = (0, 1000.0)
        
        for name, met in [("vacc_coa", vacc_coa), ("vacc_c", vacc_c),
                           ("PG", PG), ("PE_c", PE_c), ("OL", OL), ("QL", QL),
                           ("SAL", SAL)]:
            with model:
                dm = cobra.Reaction(f"DM_{name}")
                dm.add_metabolites({met: -1})
                dm.bounds = (0, 1000.0)
                model.add_reactions([dm])
                model.objective = dm.id
                sol = model.optimize()
                print(f"  Max {name:12s}: {sol.objective_value:.4f}")
    
    # =====================================================================
    # MOLAR MASS SUMMARY
    # =====================================================================
    print("\n=== MOLAR MASS SUMMARY ===")
    atomic_weights = {"C": 12.011, "H": 1.008, "N": 14.007, "O": 15.999, 
                      "P": 30.974, "S": 32.065}
    
    lipid_mets = [
        ("PG",      PG),
        ("PE",      PE_c),
        ("MMPE",    MMPE_c),
        ("DMPE",    DMPE_c),
        ("lyso-PE", lyso_PE_c),
        ("OL",      OL),
        ("QL",      QL),
        ("SAL",     SAL),
    ]
    
    for label, met in lipid_mets:
        elems = _parse_formula(met.formula)
        mw = sum(atomic_weights.get(e, 0) * n for e, n in elems.items())
        print(f"  {label:8s}  {met.formula:20s}  charge={met.charge:+d}  MW={mw:.3f} g/mol")
        
    # Verify: sum of (1/coeff * MW / 1000) should ≈ 1.0 g
    print("\n=== LIPID MASS VERIFICATION ===")
    """
    total_g = 0
    for label, coeff_C, mw in [
        ("PG_c",     0.820508, 773.042),
        ("PG_p",     0.549170, 773.042),
        ("PE_c",     10.88066, 744.048),
        ("PE_p",     4.207201, 744.048),
        ("MMPE_c",   4.004081, 758.075),
        ("MMPE_p",   1.286139, 758.075),
        ("DMPE_c",   50.05102, 772.102),
        ("DMPE_p",   15.51405, 772.102),
        ("lysoPE_c", 100.1020, 479.595),
        ("lysoPE_p", 82.74161, 479.595),
        ("OL_c",     7.470301, 678.076),
        ("OL_p",     9.547109, 678.076),
        ("QL_c",     3.653359, 691.051),
        ("QL_p",     1.021501, 691.051),
        ("SAL_p",    31.02810, 713.053),
    ]:
        contrib = (1.0/coeff_C) * mw / 1000.0
        total_g += contrib
    print(f"Total: {total_g:.4f} g")
    """ 
    total_g = 0
    for label, coeff, mw in [
        ("PG_c",      1.8461,  773.042),
        ("PG_p",      4.9425,  773.042),
        ("PE_c",     24.4815,  744.048),
        ("PE_p",     37.8648,  744.048),
        ("MMPE_c",    9.0092,  758.075),
        ("MMPE_p",   11.5753,  758.075),
        ("DMPE_c",  112.6148,  772.102),
        ("DMPE_p",  139.6265,  772.102),
        ("lysoPE_c",225.2296,  479.595),
        ("lysoPE_p",744.6745,  479.595),
        ("OL_c",     16.8082,  678.076),
        ("OL_p",     85.9240,  678.076),
        ("QL_c",      8.2201,  691.051),
        ("QL_p",      9.1935,  691.051),
        ("SAL_p",   279.2529,  713.053),
    ]:
        contrib = (1.0/coeff) * mw / 1000.0
        total_g += contrib
        print(f"  {label:12s}  coeff={coeff:>12.4f}  contrib={contrib:.6f} g")
    print(f"  TOTAL: {total_g:.4f} g  (should be ~1.0)")

=== Knocking out old C16 pathway ===
  Skip: RXN_2.3.1.15
  Skip: RXN_2.3.1.51
  KO: 1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN
  KO: CDPDIGLYSYN-RXN
  KO: CDPDIGLYSYN-RXN-2
  KO: PHOSPHASERSYN-RXN
  KO: PHOSPHASERSYN-RXN-2
  KO: PHOSPHASERDECARB-RXN
  KO: PHOSPHASERDECARB-RXN-2
  KO: 3.6.3.1-RXN
  KO: 3.6.3.1-RXN-2
  KO: PHOSPHAGLYPSYN-RXN
  KO: LIPID-RXN

=== FAS II FIXES — MASS BALANCE ===
  5.3.3.14-RXN                             OK
  3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP OK
  RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON OK
  RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c] OK

=== C18:1 GLYCEROPHOSPHOLIPID — MASS BALANCE ===
  RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A OK
  RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A OK
  CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI OK
  PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON OK
  PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXI

100%|██████████| 100/100 [00:00<00:00, 113.40it/s]


=== MOLAR MASS SUMMARY ===
  PG        C42H77O10P            charge=-2  MW=773.042 g/mol
  PE        C41H78NO8P            charge=+0  MW=744.048 g/mol
  MMPE      C42H80NO8P            charge=+0  MW=758.075 g/mol
  DMPE      C43H82NO8P            charge=+0  MW=772.102 g/mol
  lyso-PE   C23H46NO7P            charge=+0  MW=479.595 g/mol
  OL        C41H77N2O5            charge=-1  MW=678.076 g/mol
  QL        C41H74N2O6            charge=-2  MW=691.051 g/mol
  SAL       C39H70NO8S            charge=-3  MW=713.053 g/mol

=== LIPID MASS VERIFICATION ===
  PG_c          coeff=      1.8461  contrib=0.418743 g
  PG_p          coeff=      4.9425  contrib=0.156407 g
  PE_c          coeff=     24.4815  contrib=0.030392 g
  PE_p          coeff=     37.8648  contrib=0.019650 g
  MMPE_c        coeff=      9.0092  contrib=0.084145 g
  MMPE_p        coeff=     11.5753  contrib=0.065491 g
  DMPE_c        coeff=    112.6148  contrib=0.006856 g
  DMPE_p        coeff=    139.6265  contrib=0.005530 g
  l

In [6]:
oh_stearoyl_coa

Metabolite identifier,CPD-10261[c]
Name,(3R)-3-hydroxyoctadecanoyl-CoA
Memory address,0x331d8dd30
Formula,C39H66N7O18P3S
Compartment,c
In 2 reaction(s),"RXN-9545, OHBUTYRYL-COA-EPIM-RXN-CPD-10261//POLYMER-INST-L-3-HYDROXYACYL-COA-C14-H28.52."


In [15]:
ATPM_space = np.linspace(0,50,100)
growth_rates = []
growth_rates_ac = []

import re
from collections import defaultdict

def _parse_formula(formula):
    d = defaultdict(int)
    for elem, num in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        d[elem] += int(num) if num else 1
    return d

def _dict_to_formula(d):
    order = ["C","H","N","O","P","S"]
    elems = order + [e for e in sorted(d.keys()) if e not in order]
    out = []
    for e in elems:
        n = d.get(e, 0)
        if n <= 0: continue
        out.append(e if n == 1 else f"{e}{n}")
    return "".join(out)

with model:
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    model.reactions.get_by_id("EX_ac").bounds = (0, 0)
    
    # key knockouts
    model.reactions.get_by_id("METHYLENETHFDEHYDROG-NADP-RXN").knock_out()
    model.reactions.get_by_id("RXN0-300").knock_out()
    model.reactions.get_by_id("GLYCERATE-DEHYDROGENASE-RXN").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NAD/WATER//OXALACETIC_ACID/AMMONIUM/NADH/PROTON.60.").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NADP/WATER//OXALACETIC_ACID/AMMONIUM/NADPH/PROTON.62.").knock_out()
    model.reactions.get_by_id("1.4.1.21-RXN-L-ASPARTATE/NAD/WATER//OXALACETIC_ACID/AMMONIUM/NADH/PROTON.60.").knock_out()
    model.reactions.get_by_id("NADH-KINASE-RXN").knock_out()
    model.reactions.get_by_id("R501-RXN-CPD-821/NADPH/OXYGEN-MOLECULE/PROTON//CPD-904/FORMALDEHYDE/NADP/WATER.70.").knock_out()
    model.reactions.get_by_id("R501-RXN-CPD-821/NADH/OXYGEN-MOLECULE/PROTON//CPD-904/FORMALDEHYDE/NAD/WATER.68.").knock_out()
    model.reactions.get_by_id("RXN0-3962").knock_out()
    
    # =====================================================================
    # COMMON METABOLITES
    # =====================================================================
    WATER = model.metabolites.get_by_id("WATER[c]")
    Pi = model.metabolites.get_by_id("Pi[c]")
    PROTON = model.metabolites.get_by_id("PROTON[c]")
    G3P = model.metabolites.get_by_id("GLYCEROL-3P[c]")
    CTP = model.metabolites.get_by_id("CTP[c]")
    PPI = model.metabolites.get_by_id("PPI[c]")
    CMP = model.metabolites.get_by_id("CMP[c]")
    SER = model.metabolites.get_by_id("SER[c]")
    CO2 = model.metabolites.get_by_id("CARBON-DIOXIDE[c]")
    COA = model.metabolites.get_by_id("CO-A[c]")
    ATP = model.metabolites.get_by_id("ATP[c]")
    ADP = model.metabolites.get_by_id("ADP[c]")
    NADH = model.metabolites.get_by_id("NADH[c]")
    NAD = model.metabolites.get_by_id("NAD[c]")
    O2 = model.metabolites.get_by_id("OXYGEN-MOLECULE[c]")
    orn = model.metabolites.get_by_id("L-ORNITHINE[c]")
    gln = model.metabolites.get_by_id("GLN[c]")
    ACP = model.metabolites.get_by_id("ACP[c]")
    
    # Acyl-CoA donors
    stearoyl_coa = model.metabolites.get_by_id("STEAROYL-COA[c]")
    oh_stearoyl_coa = model.metabolites.get_by_id("CPD-10261[c]")  # 3R-OH-C18:0-CoA
    vacc_coa = model.metabolites.get_by_id("CPD-18346[c]")         # cis-vaccenoyl-CoA
    
    # =====================================================================
    # KNOCK OUT OLD C16 GLYCEROPHOSPHOLIPID PATHWAY
    # =====================================================================
    old_rxns_to_ko = [
        "RXN_2.3.1.15", "RXN_2.3.1.51", "1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN",
        "CDPDIGLYSYN-RXN", "CDPDIGLYSYN-RXN-2",
        "PHOSPHASERSYN-RXN", "PHOSPHASERSYN-RXN-2",
        "PHOSPHASERDECARB-RXN", "PHOSPHASERDECARB-RXN-2",
        "3.6.3.1-RXN", "3.6.3.1-RXN-2",
        "PHOSPHAGLYPSYN-RXN", "LIPID-RXN",
    ]
    print("=== Knocking out old C16 pathway ===")
    for rid in old_rxns_to_ko:
        try:
            model.reactions.get_by_id(rid).knock_out()
            print(f"  KO: {rid}")
        except KeyError:
            print(f"  Skip: {rid}")
    
    # =====================================================================
    # FAS II FIXES — connect unsaturated branch to produce C18:1
    #
    # The model's FAS II pathway has the full unsaturated chain from C10
    # to C18:1 but two reactions are missing:
    #   1. FabA isomerase 5.3.3.14-RXN: trans-2-decenoyl-ACP ⇌ cis-3-decenoyl-ACP
    #      (branch point for unsaturated FA synthesis, EC 4.2.1.60)
    #   2. FabF elongation: palmitoleoyl-ACP + malonyl-ACP → 
    #      3-oxo-cis-vaccenoyl-ACP + ACP + CO2 (EC 2.3.1.179)
    #
    # We also add a thioesterase to release cis-vaccenate from the ACP
    # cycle, and a transport to move it to periplasm where the existing
    # CoA ligase (RXN0-7238) activates it to cis-vaccenoyl-CoA.
    #
    # Net route:
    #   FAS II → trans-2-decenoyl-ACP → FabA → cis-3-decenoyl-ACP
    #   → FabB elongation → ... → palmitoleoyl-ACP
    #   → FabF elongation → ... → cis-vaccenoyl-ACP
    #   → thioesterase → cis-vaccenate[c] + ACP (recycled)
    #   → transport → cis-vaccenate[p]
    #   → CoA ligase (RXN0-7238, exists) → cis-vaccenoyl-CoA
    # =====================================================================
    
    # FIX 1: FabA isomerase (EC 4.2.1.60)
    # trans-2-decenoyl-ACP ⇌ cis-3-decenoyl-ACP
    t2d = model.metabolites.get_by_id("Trans-D2-decenoyl-ACPs[c]")
    c3d = model.metabolites.get_by_id("Cis-delta-3-decenoyl-ACPs[c]")
    
    # this exists in R Pom
    # bifunctional 3-hydroxydecanoyl-ACP dehydratase/trans-2-decenoyl-ACP isomerase
    # NCBI Reference Sequence: WP_011242221.1
    rxn_fabA = cobra.Reaction("5.3.3.14-RXN")
    rxn_fabA.name = "FabA isomerase"
    rxn_fabA.bounds = (-1000.0, 1000.0)
    rxn_fabA.add_metabolites({t2d: -1, c3d: 1})
    
    # FIX 2: FabF elongation of C16:1 → C18:1 (EC 2.3.1.179)
    # palmitoleoyl-ACP + malonyl-ACP + H+ → 3-oxo-cis-vaccenoyl-ACP + ACP + CO2
    
    # this exists in R Pom
    # beta-ketoacyl-ACP synthase II
    # NCBI Reference Sequence: WP_011047991.1
    palm_oleyl = model.metabolites.get_by_id("Palmitoleoyl-ACPs[c]")
    mal_acp = model.metabolites.get_by_id("MALONYL-ACP[c]")
    oxo_vacc = model.metabolites.get_by_id("3-oxo-cis-vaccenoyl-ACPs[c]")
    
    rxn_fabF = cobra.Reaction("3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP")
    rxn_fabF.name = "FabF C16:1→C18:1 elongation (EC 2.3.1.179)"
    rxn_fabF.bounds = (0, 1000.0)
    rxn_fabF.add_metabolites({
        palm_oleyl: -1, mal_acp: -1, PROTON: -1,
        oxo_vacc: 1, ACP: 1, CO2: 1,
    })
    
    # FIX 3: cis-vaccenoyl-ACP thioesterase (EC 3.1.2.14)
    # cis-vaccenoyl-ACP + H2O → cis-vaccenate + ACP + H+
    
    # reaction likely exists
    # acyl-CoA thioesterase
    # NCBI Reference Sequence: WP_011047689.1
    vacc_acp_met = model.metabolites.get_by_id("Cis-vaccenoyl-ACPs[c]")
    
    vacc_c = cobra.Metabolite("CPD-9247[c]", formula="C18H33O2", charge=-1,
        name="cis-vaccenate", compartment="c")
    
    rxn_vacc_TE = cobra.Reaction("RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON")
    rxn_vacc_TE.name = "cis-vaccenoyl-ACP thioesterase (EC 3.1.2.14)"
    rxn_vacc_TE.bounds = (0, 1000.0)
    rxn_vacc_TE.add_metabolites({
        vacc_acp_met: -1, WATER: -1,
        ACP: 1, vacc_c: 1, PROTON: 1,
    })
    
    # FIX 4:
    # cis-vaccenate[c] + ATP + CoA → cis-vaccenoyl-CoA + AMP + PPi
    
    # reaction likely exists
    # acyl-CoA synthetase
    # NCBI Reference Sequence: WP_011046428.1
    AMP = model.metabolites.get_by_id("AMP[c]")
    
    rxn_vacc_ligase = cobra.Reaction("RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c]")
    rxn_vacc_ligase.name = "cis-vaccenate--CoA ligase (EC 6.2.1.3)"
    rxn_vacc_ligase.bounds = (0, 1000.0)
    rxn_vacc_ligase.add_metabolites({
        vacc_c: -1, ATP: -1, COA: -1,
        vacc_coa: 1, AMP: 1, PPI: 1,
    })
    
    print("\n=== FAS II FIXES — MASS BALANCE ===")
    fas2_rxns = [rxn_fabA, rxn_fabF, rxn_vacc_TE, rxn_vacc_ligase]
    for rxn in fas2_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:40s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
    model.add_reactions(fas2_rxns)
    
    
    
    
    # =====================================================================
    # C18:1 GLYCEROPHOSPHOLIPID and PHOSPHATIDYLETHANOLAMINE PATHWAY (CoA-based)
    # https://metacyc.org/pathway?orgid=META&id=PWY4FS-8
    # Uses cis-vaccenoyl-CoA (CPD-18346) for both PlsB and PlsC.
    # All formulas derived computationally from mass balance:
    #   lyso-PA  = C21H39O7P      charge=-2
    #   PA       = C39H71O8P      charge=-2
    #   CDP-DAG  = C48H83N3O15P2  charge=-2
    #   PS       = C42H77NO10P    charge=-1
    #   PE       = C41H78NO8P     charge=0
    #   PGP      = C42H77O13P2    charge=-3
    #   PG       = C42H77O10P     charge=-2
    # =====================================================================
    
    # inferred metabolites
    lyso_PA = cobra.Metabolite("LYSO-PA-C18[c]", formula="C21H39O7P", charge=-2,
        name="1-(11Z)-vaccenoyl-sn-glycerol-3-phosphate", compartment="c")
    PA = cobra.Metabolite("PA-C18[c]", formula="C39H71O8P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-sn-glycerol-3-phosphate", compartment="c")
    CDP_DAG = cobra.Metabolite("CDP-DAG-C18[c]", formula="C48H83N3O15P2", charge=-2,
        name="CDP-1,2-di-(11Z)-vaccenoyl-glycerol", compartment="c")
    PS = cobra.Metabolite("PS-C18[c]", formula="C42H77NO10P", charge=-1,
        name="1,2-di-(11Z)-vaccenoyl-phosphatidylserine", compartment="c")
    PE_c = cobra.Metabolite("PE-C18[c]", formula="C41H78NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-PE", compartment="c")
    PE_p = cobra.Metabolite("PE-C18[p]", formula="C41H78NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-PE", compartment="p")
    PGP_ = cobra.Metabolite("PGP-C18[c]", formula="C42H77O13P2", charge=-3,
        name="1,2-di-(11Z)-vaccenoyl-PGP", compartment="c")
    PG = cobra.Metabolite("PG-C18[c]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="c")
    PG_e = cobra.Metabolite("PG-C18[e]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="e")
    
    # RXN-1381 
    # an acyl-CoA + sn-glycerol 3-phosphate → a 2-lysophosphatidate + coenzyme A
    rxn_plsB = cobra.Reaction("RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A")
    rxn_plsB.name = "RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A"
    rxn_plsB.bounds = (0, 1000.0)
    rxn_plsB.add_metabolites({G3P: -1, vacc_coa: -1, lyso_PA: 1, COA: 1})
    
    rxn_plsC = cobra.Reaction("RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A")
    rxn_plsC.name = "RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A"
    rxn_plsC.bounds = (0, 1000.0)
    rxn_plsC.add_metabolites({lyso_PA: -1, vacc_coa: -1, PA: 1, COA: 1})
    
    rxn_cdsA = cobra.Reaction("CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI")
    rxn_cdsA.name = "CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI"
    rxn_cdsA.bounds = (0, 1000.0)
    rxn_cdsA.add_metabolites({PA: -1, CTP: -1, PROTON: -1, CDP_DAG: 1, PPI: 1})
    
    # ============================
    # Phosphatidylethanolamine synthesis leveraging same pathway from before 
    # using phosphatidylserine precursor but now using 18 carbons 
    rxn_pssA = cobra.Reaction("PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON")
    rxn_pssA.name = "PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON"
    rxn_pssA.bounds = (0, 1000.0)
    rxn_pssA.add_metabolites({CDP_DAG: -1, SER: -1, PS: 1, CMP: 1, PROTON: 1})
    
    rxn_psd = cobra.Reaction("PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE")
    rxn_psd.name = "PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE"
    rxn_psd.bounds = (0, 1000.0)
    rxn_psd.add_metabolites({PS: -1, PROTON: -1, PE_c: 1, CO2: 1})
    
    rxn_flip = cobra.Reaction("3.6.3.1-RXN-PE-C18[c]/ATP/WATER//PE-C18[p]/ADP/PROTON/Pi")
    rxn_flip.name = "3.6.3.1-RXN-PE-C18[c]/ATP/WATER//PE-C18[p]/ADP/PROTON/Pi"
    rxn_flip.bounds = (0, 1000.0)
    rxn_flip.add_metabolites({ATP: -1, PE_c: -1, WATER: -1,
                               ADP: 1, PE_p: 1, PROTON: 1, Pi: 1})
    
    # ============================
    # phosphatidyl glycerol synthesis continued
    rxn_pgsA = cobra.Reaction("PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON")
    rxn_pgsA.name = "PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON"
    rxn_pgsA.bounds = (0, 1000.0)
    rxn_pgsA.add_metabolites({CDP_DAG: -1, G3P: -1, PGP_: 1, CMP: 1, PROTON: 1})
    
    rxn_pgp = cobra.Reaction("PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON")
    rxn_pgp.name = "PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON"
    rxn_pgp.bounds = (0, 1000.0)
    rxn_pgp.add_metabolites({PGP_: -1, WATER: -1, PG: 1, Pi: 1, PROTON: 1})
    
    glycerophospholipid_rxns = [rxn_plsB, rxn_plsC, rxn_cdsA, rxn_pssA,
                                 rxn_psd, rxn_flip, rxn_pgsA, rxn_pgp]
    model.add_reactions(glycerophospholipid_rxns)
    
    print("\n=== C18:1 GLYCEROPHOSPHOLIPID — MASS BALANCE ===")
    for rxn in glycerophospholipid_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:35s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    # =====================================================================
    # ORNITHINE LIPID (OL) — mixed chains: 3-OH-C18:0 (amide) + C18:1 (ester)
    # MetaCyc PWY-6818. R. pomeroyi: SPO1980=OlsB, SPO1979=OlsA
    # References: Weissenmayer 2002, Gao 2004, Smith 2019, Stirrup 2022
    #
    # OlsB uses 3-OH-stearoyl-CoA (CPD-10261, producible, C18:0).
    # OlsA uses vaccenoyl-CoA (CPD-18346, C18:1).
    # Mixed chains (saturated amide + unsaturated ester) common in bacteria.
    #
    # Formulas from mass balance:
    #   lyso-OL = orn + CPD-10261 - CoA - H+ = C23H46N2O4, charge 0
    #   OL = lyso-OL + CPD-18346 - CoA = C41H78N2O5, charge 0
    
    # SEE WRAP_Theses_Silvano_2021.pdf
    # https://www.nature.com/articles/s41396-018-0249-z/figures/2
    # =====================================================================
    
    # =====================================================================
    # FREE 3-HYDROXY-STEARIC ACID (new metabolite)
    # Released from 3-OH-stearoyl-CoA by thioesterase
    # Formula: CPD-10261 (C39H66N7O18P3S) + H2O - CoA (C21H32N7O16P3S) - H+
    #        = C18H35O3, charge -1
    # =====================================================================
    
    OH_STEARATE = cobra.Metabolite("3R-OH-STEARATE[c]",
        formula="C18H35O3", charge=-1,
        name="(3R)-3-hydroxystearate (free acid)", compartment="c")
    lyso_OL = cobra.Metabolite("LYSO-OL-C23[c]",
        formula="C23H46N2O4", charge=0,
        name="(2R)-5-amino-2-(3-hydroxyoctadecanoylamino)pentanoic acid", compartment="c")
    OL = cobra.Metabolite("OL-LIPID-C41[c]",
        formula="C41H77N2O5", charge=-1,
        name="ornithine lipid (3-OH-C18:0 amide, C18:1 ester)", compartment="c")
    
    # Thioesterase: 3-OH-stearoyl-CoA + H2O → 3-OH-stearate + CoA + H+
    # Analogous to RXN-9624 (stearoyl-CoA thioesterase)
    # likely real reaction
    # Gene: WP_011047689.1 (acyl-CoA thioesterase, broad specificity)
    rxn_oh_stearate_TE = cobra.Reaction("THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON")
    rxn_oh_stearate_TE.name = "THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON"
    rxn_oh_stearate_TE.bounds = (0, 1000.0)
    rxn_oh_stearate_TE.add_metabolites({
        oh_stearoyl_coa: -1, WATER: -1,
        OH_STEARATE: 1, COA: 1, PROTON: 1,
    })
    
    model.add_reactions([rxn_oh_stearate_TE])
    bal = rxn_oh_stearate_TE.check_mass_balance()
    print(f"  Thioesterase mass balance: {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    rxn_olsB = cobra.Reaction("RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER")
    rxn_olsB.name = "RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER"
    rxn_olsB.bounds = (0, 1000.0)
    rxn_olsB.add_metabolites({
        orn: -1, OH_STEARATE: -1,
        lyso_OL: 1, WATER: 1,
    })
    
    rxn_olsA_OL = cobra.Reaction("RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER")
    rxn_olsA_OL.name = "RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER"
    rxn_olsA_OL.bounds = (0, 1000.0)
    rxn_olsA_OL.add_metabolites({
        lyso_OL: -1, vacc_c: -1,
        OL: 1, WATER: 1,
    })
    
    # =====================================================================
    # GLUTAMINE LIPID (QL) — free fatty acid route
    # GlsB: glutamine + 3-OH-stearate → lyso-QL + H2O
    # OlsA: lyso-QL + cis-vaccenate → QL + H2O
    # see https://www.nature.com/articles/s41396-018-0249-z/figures/2
    # =====================================================================
    
    lyso_QL = cobra.Metabolite("LYSO-QL-C23[c]",
        formula="C23H43N2O5", charge=-1,
        name="lyso-QL (N-(3R-hydroxyoctadecanoyl)-L-glutamine)", compartment="c")
    
    QL = cobra.Metabolite("QL-LIPID-C41[c]",
        formula="C41H74N2O6", charge=-2,
        name="glutamine lipid (3-OH-C18:0 amide, C18:1 ester)", compartment="c")
    
    rxn_glsB = cobra.Reaction("RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER")
    rxn_glsB.name = "RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER"
    rxn_glsB.bounds = (0, 1000.0)
    rxn_glsB.add_metabolites({
        gln: -1, OH_STEARATE: -1,
        lyso_QL: 1, WATER: 1,
    })
    
    rxn_olsA_QL = cobra.Reaction("RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER")
    rxn_olsA_QL.name = "RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER"
    rxn_olsA_QL.bounds = (0, 1000.0)
    rxn_olsA_QL.add_metabolites({
        lyso_QL: -1, vacc_c: -1,
        QL: 1, WATER: 1,
    })
    
    aminolipid_rxns = [rxn_olsB, rxn_olsA_OL, rxn_glsB, rxn_olsA_QL]
    model.add_reactions(aminolipid_rxns)
    
    print("\n=== AMINOLIPID (free acid route) — MASS BALANCE ===")
    for rxn in [rxn_oh_stearate_TE] + aminolipid_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:35s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    # =====================================================================
    # SAL — SULFUR-CONTAINING AMINOLIPID
    # Smith et al. 2021 ISME J 15:2440-53. SalA=SPO0716.
    # Silvano 2021 thesis Fig 4.7 (pathway diagram).
    #
    # Headgroup = aminopropanesulfonate (APS; homotaurine or 2-APS)
    # Lumped APS synthesis from homocysteine.
    # SalB (unknown N-acyltransferase): APS + 3-OH-stearoyl-CoA → lyso-SAL
    # SalA (SPO0716): lyso-SAL + vaccenoyl-CoA → SAL
    # Mixed chains like OL/QL.
    # =====================================================================
    
    CYSTEATE = model.metabolites.get_by_id("L-CYSTEATE[c]")
    
    LYSO_SAL = cobra.Metabolite("LYSO-SAL[c]",
        formula="C21H39NO7S", charge=-2,
        name="lyso-SAL (N-(3R-hydroxyoctadecanoyl)-L-cysteate)", compartment="c")
    
    SAL = cobra.Metabolite("SAL-LIPID[c]",
        formula="C39H70NO8S", charge=-3,
        name="SAL (3-OH-C18:0 amide, C18:1 ester, cysteate headgroup)", compartment="c")
    
    # SalB: cysteate + 3-OH-stearate → lyso-SAL + H2O
    rxn_salB = cobra.Reaction("RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER")
    rxn_salB.name = "RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER (unknown enzyme)"
    rxn_salB.bounds = (0, 1000.0)
    rxn_salB.add_metabolites({
        CYSTEATE: -1, OH_STEARATE: -1,
        LYSO_SAL: 1, WATER: 1,
    })
    
    # SalA: lyso-SAL + cis-vaccenate → SAL + H2O
    rxn_salA = cobra.Reaction("RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER")
    rxn_salA.name = "RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER (SPO0716)"
    rxn_salA.bounds = (0, 1000.0)
    rxn_salA.add_metabolites({
        LYSO_SAL: -1, vacc_c: -1,
        SAL: 1, WATER: 1,
    })
    
    sal_rxns = [rxn_salB, rxn_salA] 
    model.add_reactions(sal_rxns)
    
    print("\n=== SAL PATHWAY — MASS BALANCE ===")
    for rxn in sal_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
    # =====================================================================
    # N-METHYLATION PATHWAY
    # For DMPE and MMPE generation
    # =====================================================================
    
    # PE N-methylation pathway (PmtA, EC 2.1.1.17)
    # R. pomeroyi accumulates MMPE and DMPE (Stirrup 2022)
    # Uses SAM as methyl donor, produces SAH
    
    SAM = model.metabolites.get_by_id("S-ADENOSYLMETHIONINE[c]")
    SAH = model.metabolites.get_by_id("ADENOSYL-HOMO-CYS[c]")
    
    # MMPE = PE + CH3 (from SAM) - H (replaced on nitrogen)
    # PE:   C41H78NO8P, charge=0
    # +CH3, -H = +CH2 = +C +2H
    # MMPE: C42H80NO8P, charge=0
    MMPE_c = cobra.Metabolite("MMPE-C18[c]",
        formula="C42H80NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-MMPE", compartment="c")
    
    # DMPE = MMPE + CH2
    # DMPE: C43H82NO8P, charge=0
    DMPE_c = cobra.Metabolite("DMPE-C18[c]",
        formula="C43H82NO8P", charge=0,
        name="1,2-di-(11Z)-vaccenoyl-DMPE", compartment="c")
    
    # Step 1: PE + SAM → MMPE + SAH + H+
    rxn_pmt1 = cobra.Reaction("2.1.1.17-RXN-PE-C18/SAM//MMPE-C18/SAH/PROTON")
    rxn_pmt1.name = "2.1.1.17-RXN-PE-C18/SAM//MMPE-C18/SAH/PROTON"
    # gene SPOA0294
    rxn_pmt1.bounds = (0, 1000.0)
    rxn_pmt1.add_metabolites({
        PE_c: -1, SAM: -1,
        MMPE_c: 1, SAH: 1, PROTON: 1,
    })
    # Step 2: MMPE + SAM → DMPE + SAH + H+
    rxn_pmt2 = cobra.Reaction("2.1.1.71-RXN-MMPE-C18/SAM//DMPE-C18/SAH/PROTON")
    rxn_pmt2.name = "2.1.1.71-RXN-MMPE-C18/SAM//DMPE-C18/SAH/PROTON"
    rxn_pmt2.bounds = (0, 1000.0)
    rxn_pmt2.add_metabolites({
        MMPE_c: -1, SAM: -1,
        DMPE_c: 1, SAH: 1, PROTON: 1,
    })
    
    n_methylation_rxns = [rxn_pmt1, rxn_pmt2]
    model.add_reactions(n_methylation_rxns)
    
    print("\n=== N-METHYLATION PATHWAY — MASS BALANCE ===")
    for rxn in n_methylation_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
       
    # =====================================================================
    # LIPID TRANSPORT — inner membrane to outer membrane
    # MsbA (EC 3.6.3.1) flips lipids across the inner membrane.
    # Consistent with existing model reactions 3.6.3.1-RXN and 3.6.3.1-RXN-2
    # which use: lipid[c] + ATP + H2O → lipid[p] + ADP + Pi + H+
    #
    # We already have PE[c]→[p] flippase (rxn_flip).
    # Add PG, OL, QL, SAL flippases using the same stoichiometry.
    # 
    # Gene: MsbA homolog likely present in R. pomeroyi (essential in 
    # gram-negatives). MsbA has broad substrate specificity for lipids.
    # =====================================================================
    
    MMPE_p = cobra.Metabolite("MMPE-C18[p]", formula=MMPE_c.formula, charge=MMPE_c.charge,
        name=MMPE_c.name, compartment="p")
    DMPE_p = cobra.Metabolite("DMPE-C18[p]", formula=DMPE_c.formula, charge=DMPE_c.charge,
        name=DMPE_c.name, compartment="p")
    
    rxn_flip_m = cobra.Reaction("3.6.3.1-RXN-MMPE-C18[c]/ATP/WATER//MMPE-C18[p]/ADP/PROTON/Pi")
    rxn_flip_m.name = "3.6.3.1-RXN-MMPE-C18[c]/ATP/WATER//MMPE-C18[p]/ADP/PROTON/Pi"
    rxn_flip_m.bounds = (0, 1000.0)
    rxn_flip_m.add_metabolites({ATP: -1, MMPE_c: -1, WATER: -1,
                               ADP: 1, MMPE_p: 1, PROTON: 1, Pi: 1})
    
    rxn_flip_d = cobra.Reaction("3.6.3.1-RXN-DMPE-C18[c]/ATP/WATER//DMPE-C18[p]/ADP/PROTON/Pi")
    rxn_flip_d.name = "3.6.3.1-RXN-DMPE-C18[c]/ATP/WATER//DMPE-C18[p]/ADP/PROTON/Pi"
    rxn_flip_d.bounds = (0, 1000.0)
    rxn_flip_d.add_metabolites({ATP: -1, DMPE_c: -1, WATER: -1,
                               ADP: 1, DMPE_p: 1, PROTON: 1, Pi: 1})
    
    # Metacyc derived
    
    # PG flippase [c] → [p]
    PG_p = cobra.Metabolite("PG-C18[p]", formula="C42H77O10P", charge=-2,
        name="1,2-di-(11Z)-vaccenoyl-PG", compartment="p")
        
    rxn_pg_flip = cobra.Reaction("TRANS-RXN-1533-PG-C18[c]/PG-C18[e]")
    # no known enzyme
    rxn_pg_flip.name = "TRANS-RXN-1533-PG-C18[c]/PG-C18[e]"
    rxn_pg_flip.bounds = (0, 1000.0)
    rxn_pg_flip.add_metabolites({
        PG: -1,
        PG_p: 1
    })
    
    # OL flippase [c] → [p]
    OL_p = cobra.Metabolite("OL-LIPID-C41[p]", formula=OL.formula, charge=OL.charge,
        name=OL.name, compartment="p")
    # yddG gene in E. Coli.
    # but not known in R Pom. Yddg BLAST returns no results.
    rxn_ol_flip = cobra.Reaction("TRANS-RXN0-265-OL-C41[c]//OL-C41[p]")
    rxn_ol_flip.name = "TRANS-RXN0-265-OL-C41[c]//OL-C41[p]"
    rxn_ol_flip.bounds = (0, 1000.0)
    rxn_ol_flip.add_metabolites({
        OL: -1,
        OL_p: 1
    })
    
    # QL flippase [c] → [p]
    QL_p = cobra.Metabolite("QL-LIPID-C41[p]", formula=QL.formula, charge=QL.charge,
        name=QL.name, compartment="p")
    
    rxn_ql_flip = cobra.Reaction("TRANS-RXN0-265-QL-C41[c]//QL-C41[p]")
    rxn_ql_flip.name = "TRANS-RXN0-265-QL-C41[c]//QL-C41[p]"
    rxn_ql_flip.bounds = (0, 1000.0)
    rxn_ql_flip.add_metabolites({
        QL: -1, 
        QL_p: 1
    })
    
    # SAL flippase [c] → [p]
    SAL_p = cobra.Metabolite("SAL-LIPID[p]", formula=SAL.formula, charge=SAL.charge,
        name=SAL.name, compartment="p")
    
    rxn_sal_flip = cobra.Reaction("TRANS-RXN0-265-SAL[c]//SAL[p]")
    rxn_sal_flip.name = "TRANS-RXN0-265-SAL[c]//SAL[p]"
    rxn_sal_flip.bounds = (0, 1000.0)
    rxn_sal_flip.add_metabolites({
        SAL: -1,
        SAL_p: 1
    })
    
    flippase_rxns = [rxn_pg_flip, rxn_ol_flip, rxn_ql_flip, rxn_sal_flip, rxn_flip_m, rxn_flip_d]
    model.add_reactions(flippase_rxns)
    
    print("\n=== FLIP REACTIONS — MASS BALANCE ===")
    for rxn in flippase_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
        
        
    # =====================================================================
    # LYSO-PE SYNTHESIS PATHWAY
    # =====================================================================
    WATER_p = model.metabolites.get_by_id("WATER[p]")
    PROTON_p = model.metabolites.get_by_id("PROTON[p]")
    
    vacc_p = cobra.Metabolite("CPD-9247[p]", formula="C18H33O2", charge=-1,
        name="cis-vaccenate", compartment="p")
    
    lyso_PE_p = cobra.Metabolite("LYSO-PE-C18[p]",
        formula="C23H46NO7P", charge=0,
        name="1-(11Z)-vaccenoyl-sn-glycero-3-phosphoethanolamine", compartment="p")
    
    lyso_PE_c = cobra.Metabolite("LYSO-PE-C18[c]",
        formula="C23H46NO7P", charge=0,
        name="1-(11Z)-vaccenoyl-sn-glycero-3-phosphoethanolamine", compartment="c")
    
    # PldA: outer membrane phospholipase A (EC 3.1.1.32)
    # PE[p] + H2O[p] → lyso-PE[p] + vaccenate[p] + H+[p]
    # not in biocyc
    # e coli pldA
    rxn_pldA = cobra.Reaction("RXN0-6952-PE-C18[p]/WATER[p]//LYSO-PE-C18[p]/CPD-9247[p]/PROTON[p]")
    rxn_pldA.name = "RXN0-6952-PE-C18[p]/WATER[p]//LYSO-PE-C18[p]/CPD-9247[p]/PROTON[p]"
    rxn_pldA.bounds = (0, 1000.0)
    rxn_pldA.add_metabolites({
        PE_p: -1, WATER_p: -1,
        lyso_PE_p: 1, vacc_p: 1, PROTON_p: 1,
    })
    
    # LplT: lysophospholipid transporter (energy-independent)
    # lyso-PE[p] → lyso-PE[c]
    # not in biocyc
    # e coli lplT
    rxn_lplT = cobra.Reaction("TRANS-RXN-294-LYSO-PE-C18[p]//LYSO-PE-C18[c]")
    rxn_lplT.name = "TRANS-RXN-294-LYSO-PE-C18[p]//LYSO-PE-C18[c]"
    rxn_lplT.bounds = (0, 1000.0)
    rxn_lplT.add_metabolites({
        lyso_PE_p: -1,
        lyso_PE_c: 1,
    })
    
    lyso_pe_rxns = [rxn_pldA, rxn_lplT]
    model.add_reactions(lyso_pe_rxns)
    
    print("\n=== LYSO-PE PATHWAY (PldA/LplT/Aas) — MASS BALANCE ===")
    for rxn in lyso_pe_rxns:
        bal = rxn.check_mass_balance()
        print(f"  {rxn.id:55s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    
    
    # =====================================================================
    # LIPID BIOMASS PSEUDOREACTION
    # =====================================================================
    
    lipid_rxn = cobra.Reaction("LIPID-RXN-RPOM")
    lipid_rxn.name = "Lipid pseudoreaction R. pomeroyi (Stirrup 2022, IM/OM resolved)"
    lipid_rxn.bounds = (0, 1000.0)
    """
    lipid_rxn.add_metabolites({
        PG:         -0.820508499,
        PG_p:       -0.549169986,
        PE_c:       -10.88065619,
        PE_p:       -4.207200571,
        MMPE_c:     -4.004081477,
        MMPE_p:     -1.286139035,
        DMPE_c:     -50.05101847,
        DMPE_p:     -15.5140521,
        lyso_PE_c:  -100.1020369,
        lyso_PE_p:  -82.74161122,
        OL:         -7.470301264,
        OL_p:       -9.547108987,
        QL:         -3.653359012,
        QL_p:       -1.021501373,
        SAL:        -0.0,
        SAL_p:      -31.02810421,
        undec:      -0.000573,
        lipid_met:   1.0,
    })
    """
    lipid_rxn.add_metabolites({
        PG:         -1.8461**-1,
        PG_p:       -4.9425**-1,
        PE_c:       -24.4815**-1,
        PE_p:       -37.8648**-1,
        MMPE_c:     -9.0092**-1,
        MMPE_p:     -11.5753**-1,
        DMPE_c:     -112.6148**-1,
        DMPE_p:     -139.6265**-1,
        lyso_PE_c:  -225.2296**-1,
        lyso_PE_p:  -744.6745**-1,
        OL:         -16.8082**-1,
        OL_p:       -85.9240**-1,
        QL:         -8.2201**-1,
        QL_p:       -9.1935**-1,
        # SAL_c: 0
        SAL_p:      -279.2529**-1,
        lipid_met:   1.0,
    })
    model.add_reactions([lipid_rxn])
    
    print(f"\n=== LIPID-RXN-RPOM (IM/OM resolved) ===")
    print(f"  {'Species':12s} {'IM ([c])':>12s} {'OM ([p])':>12s} {'Total':>12s}")
    print(f"  {'-'*48}")
    for label, met_c, met_p, coeff_c, coeff_p in [
        ("PG",      PG,        PG_p,       0.820508499, 0.549169986),
        ("PE",      PE_c,      PE_p,       10.88065619,  4.207200571),
        ("MMPE",    MMPE_c,    MMPE_p,     4.004081477,  1.286139035),
        ("DMPE",    DMPE_c,    DMPE_p,     50.05101847,  15.5140521),
        ("lyso-PE", lyso_PE_c, lyso_PE_p,  100.1020369,  82.74161122),
        ("OL",      OL,        OL_p,       7.470301264,  9.547108987),
        ("QL",      QL,        QL_p,       3.653359012,  1.021501373),
        ("SAL",     SAL,       SAL_p,      0.0,          31.02810421),
    ]:
        print(f"  {label:12s} {coeff_c:>12.4f} {coeff_p:>12.4f} {coeff_c+coeff_p:>12.4f}")    
    
    # =====================================================================
    # Fix NADH dehydrogenase (e → p)
    # =====================================================================
    model.reactions.get_by_id("NADH-DEHYDROG-A-RXN").knock_out()
    rxn_ndh = cobra.Reaction("NADH-DEHYDROG-A-RXN_2")
    rxn_ndh.add_metabolites({
        NADH: -1, model.metabolites.get_by_id("PROTON[c]"): -5,
        model.metabolites.get_by_id("UBIQUINONE-10[c]"): -1,
        model.metabolites.get_by_id("CPD-9958[c]"): 1, NAD: 1,
        model.metabolites.get_by_id("PROTON[p]"): 4,
    })
    rxn_ndh.bounds = (-1000, 1000)
    model.add_reactions([rxn_ndh])
    
    # =====================================================================
    # FULL VALIDATION
    # =====================================================================
    print("\n=== FULL MASS BALANCE CHECK ===")
    all_new = fas2_rxns + glycerophospholipid_rxns + aminolipid_rxns + sal_rxns
    all_ok = True
    for rxn in all_new:
        bal = rxn.check_mass_balance()
        if bal: all_ok = False
        print(f"  {rxn.id:45s} {'OK' if not bal else f'UNBALANCED: {bal}'}")
    print(f"\n  {'ALL BALANCED!' if all_ok else 'WARNING: imbalances'}")
    
    # =====================================================================
    # TEST GROWTH
    # =====================================================================
    model.objective = "Rpom_hwa_biomass"
    
    model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
    model.reactions.get_by_id("EX_ac").bounds = (0, 0)
    sol_glc = model.optimize()
    print(f"\nGlucose growth: {sol_glc.objective_value:.4f}")
    
    model.reactions.get_by_id("EX_glc").bounds = (0, 0)
    model.reactions.get_by_id("EX_ac").bounds = (-15.01, 0)
    sol_ac = model.optimize()
    print(f"Acetate growth:  {sol_ac.objective_value:.4f}")
    
    if sol_glc.objective_value > 0 and sol_ac.objective_value > 0:
        print(f"Ratio (glc/ac):  {sol_glc.objective_value / sol_ac.objective_value:.2f}")
        
        model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
        model.reactions.get_by_id("EX_ac").bounds = (0, 0)
        sol = model.optimize()
        print(f"\n=== Key lipid fluxes (glucose) ===")
        for rid in [
            "5.3.3.14-RXN",
            "3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP",
            "RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON",
            "RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c]",
            "RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A",
            "RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A",
            "CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI",
            "PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON",
            "PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXIDE",
            "PHOSPHAGLYPSYN-RXN-CDP-DAG-C18/GLYCEROL-3P//PGP-C18/CMP/PROTON",
            "PGPPHOSPHA-RXN-PGP-C18/WATER//PG-C18/Pi/PROTON",
            "THIOESTER-RXN-CPD-10261/WATER//3R-OH-STEARATE/COA/PROTON",
            "RXN-OLSB-PUTATIVE-L-ORNITHINE/3R-OH-STEARATE//LYSO-OL-C23/WATER",
            "RXN-OLSA-PUTATIVE-LYSO-OL-C23/CPD-9247//OL-LIPID-C41/WATER",
            "RXN-GLSB-PUTATIVE-GLN/3R-OH-STEARATE//LYSO-QL-C23/WATER",
            "RXN-OLSA-PUTATIVE-LYSO-QL-C23/CPD-9247//QL-LIPID-C41/WATER",
            "RXN-SALB-PUTATIVE-CYSTEATE/3R-OH-STEARATE//LYSO-SAL-C21/WATER",
            "RXN-SALA-PUTATIVE-LYSO-SAL-C21/CPD-9247//SAL-LIPID-C39/WATER",
        ]:
            print(f"  {rid[:50]:50s} {sol.fluxes[rid]:.6f}")
        
        # ATPM scans
        for level in tqdm(ATPM_space):
            model.reactions.get_by_id("ATPM").bounds = (level, level)
            model.reactions.get_by_id("EX_ac").bounds = (0, 0)
            model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
            sol = model.optimize()
            growth_rates.append(sol.objective_value)
        
        for level in tqdm(ATPM_space):
            model.reactions.get_by_id("ATPM").bounds = (level, level)
            model.reactions.get_by_id("EX_glc").bounds = (0, 0)
            model.reactions.get_by_id("EX_ac").bounds = (-15.01, 0)
            sol = model.optimize()
            growth_rates_ac.append(sol.objective_value)
    else:
        print("\n=== ZERO GROWTH DEBUG ===")
        model.reactions.get_by_id("EX_glc").bounds = (-5.44, 0)
        model.reactions.get_by_id("EX_ac").bounds = (0, 0)
        
        lipid_rxn.bounds = (0, 0)
        sol = model.optimize()
        print(f"  Without lipid req: {sol.objective_value:.4f}")
        lipid_rxn.bounds = (0, 1000.0)
        
        for name, met in [("vacc_coa", vacc_coa), ("vacc_c", vacc_c),
                           ("PG", PG), ("PE_c", PE_c), ("OL", OL), ("QL", QL),
                           ("SAL", SAL)]:
            with model:
                dm = cobra.Reaction(f"DM_{name}")
                dm.add_metabolites({met: -1})
                dm.bounds = (0, 1000.0)
                model.add_reactions([dm])
                model.objective = dm.id
                sol = model.optimize()
                print(f"  Max {name:12s}: {sol.objective_value:.4f}")
    
    # =====================================================================
    # MOLAR MASS SUMMARY
    # =====================================================================
    print("\n=== MOLAR MASS SUMMARY ===")
    atomic_weights = {"C": 12.011, "H": 1.008, "N": 14.007, "O": 15.999, 
                      "P": 30.974, "S": 32.065}
    
    lipid_mets = [
        ("PG",      PG),
        ("PE",      PE_c),
        ("MMPE",    MMPE_c),
        ("DMPE",    DMPE_c),
        ("lyso-PE", lyso_PE_c),
        ("OL",      OL),
        ("QL",      QL),
        ("SAL",     SAL),
    ]
    
    for label, met in lipid_mets:
        elems = _parse_formula(met.formula)
        mw = sum(atomic_weights.get(e, 0) * n for e, n in elems.items())
        print(f"  {label:8s}  {met.formula:20s}  charge={met.charge:+d}  MW={mw:.3f} g/mol")
        
    # Verify: sum of (1/coeff * MW / 1000) should ≈ 1.0 g
    print("\n=== LIPID MASS VERIFICATION ===")
    """
    total_g = 0
    for label, coeff_C, mw in [
        ("PG_c",     0.820508, 773.042),
        ("PG_p",     0.549170, 773.042),
        ("PE_c",     10.88066, 744.048),
        ("PE_p",     4.207201, 744.048),
        ("MMPE_c",   4.004081, 758.075),
        ("MMPE_p",   1.286139, 758.075),
        ("DMPE_c",   50.05102, 772.102),
        ("DMPE_p",   15.51405, 772.102),
        ("lysoPE_c", 100.1020, 479.595),
        ("lysoPE_p", 82.74161, 479.595),
        ("OL_c",     7.470301, 678.076),
        ("OL_p",     9.547109, 678.076),
        ("QL_c",     3.653359, 691.051),
        ("QL_p",     1.021501, 691.051),
        ("SAL_p",    31.02810, 713.053),
    ]:
        contrib = (1.0/coeff_C) * mw / 1000.0
        total_g += contrib
    print(f"Total: {total_g:.4f} g")
    """ 
    total_g = 0
    for label, coeff, mw in [
        ("PG_c",      1.8461,  773.042),
        ("PG_p",      4.9425,  773.042),
        ("PE_c",     24.4815,  744.048),
        ("PE_p",     37.8648,  744.048),
        ("MMPE_c",    9.0092,  758.075),
        ("MMPE_p",   11.5753,  758.075),
        ("DMPE_c",  112.6148,  772.102),
        ("DMPE_p",  139.6265,  772.102),
        ("lysoPE_c",225.2296,  479.595),
        ("lysoPE_p",744.6745,  479.595),
        ("OL_c",     16.8082,  678.076),
        ("OL_p",     85.9240,  678.076),
        ("QL_c",      8.2201,  691.051),
        ("QL_p",      9.1935,  691.051),
        ("SAL_p",   279.2529,  713.053),
    ]:
        contrib = (1.0/coeff) * mw / 1000.0
        total_g += contrib
        print(f"  {label:12s}  coeff={coeff:>12.4f}  contrib={contrib:.6f} g")
    print(f"  TOTAL: {total_g:.4f} g  (should be ~1.0)")

=== Knocking out old C16 pathway ===
  Skip: RXN_2.3.1.15
  Skip: RXN_2.3.1.51
  KO: 1-ACYLGLYCEROL-3-P-ACYLTRANSFER-RXN
  KO: CDPDIGLYSYN-RXN
  KO: CDPDIGLYSYN-RXN-2
  KO: PHOSPHASERSYN-RXN
  KO: PHOSPHASERSYN-RXN-2
  KO: PHOSPHASERDECARB-RXN
  KO: PHOSPHASERDECARB-RXN-2
  KO: 3.6.3.1-RXN
  KO: 3.6.3.1-RXN-2
  KO: PHOSPHAGLYPSYN-RXN
  KO: LIPID-RXN

=== FAS II FIXES — MASS BALANCE ===
  5.3.3.14-RXN                             OK
  3-OXOACYL-ACP-SYNTH-RXN-Palmitoleoyl-ACPs/MALONYL-ACP/PROTON//3-OXO-CIS-VACCENOYL-ACPS/CARBON-DIOXIDE/ACP OK
  RXN-21818-Cis-vaccenoyl-ACPs/WATER//ACP/CPD-9247/PROTON OK
  RXN-19960-CPD-9247[c]/ATP[c]/COA[c]//CPD-18346[c]/AMP[c]/PPI[c] OK

=== C18:1 GLYCEROPHOSPHOLIPID — MASS BALANCE ===
  RXN-1381-GLYCEROL-3P/CPD-18346//LYSO-PA-C18/CO-A OK
  RXN-1623-LYSO-PA-C18/CPD-18346//PA-C18/CO-A OK
  CDPDIGLYSYN-RXN-PA-C18/CTP/PROTON//CDP-DAG-C18/PPI OK
  PHOSPHASERSYN-RXN-CDP-DAG-C18/SER//PS-C18/CMP/PROTON OK
  PHOSPHASERDECARB-RXN-PS-C18/PROTON//PE-C18/CARBON-DIOXI

100%|██████████| 100/100 [00:00<00:00, 111.27it/s]


=== MOLAR MASS SUMMARY ===
  PG        C42H77O10P            charge=-2  MW=773.042 g/mol
  PE        C41H78NO8P            charge=+0  MW=744.048 g/mol
  MMPE      C42H80NO8P            charge=+0  MW=758.075 g/mol
  DMPE      C43H82NO8P            charge=+0  MW=772.102 g/mol
  lyso-PE   C23H46NO7P            charge=+0  MW=479.595 g/mol
  OL        C41H77N2O5            charge=-1  MW=678.076 g/mol
  QL        C41H74N2O6            charge=-2  MW=691.051 g/mol
  SAL       C39H70NO8S            charge=-3  MW=713.053 g/mol

=== LIPID MASS VERIFICATION ===
  PG_c          coeff=      1.8461  contrib=0.418743 g
  PG_p          coeff=      4.9425  contrib=0.156407 g
  PE_c          coeff=     24.4815  contrib=0.030392 g
  PE_p          coeff=     37.8648  contrib=0.019650 g
  MMPE_c        coeff=      9.0092  contrib=0.084145 g
  MMPE_p        coeff=     11.5753  contrib=0.065491 g
  DMPE_c        coeff=    112.6148  contrib=0.006856 g
  DMPE_p        coeff=    139.6265  contrib=0.005530 g
  l

In [26]:
model.genes.get_by_id("SPOA0294")

KeyError: 'SPOA0294'